Doing Genetics project

In [ ]:
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.metrics import confusion_matrix

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
genetics = pd.read_parquet("https://lab.cs307.org/genetics/data/genetics.parquet")

In [3]:
genetics

,cancer,gene_0,gene_1,gene_2,gene_3,gene_4,gene_5,gene_6,gene_7,gene_8,...,gene_3990,gene_3991,gene_3992,gene_3993,gene_3994,gene_3995,gene_3996,gene_3997,gene_3998,gene_3999
0,PRAD,0.0,2.017209,3.265527,5.478487,10.431999,0.0,7.175175,0.591871,0.0,...,0.000000,11.722884,11.831470,7.785282,11.877556,4.926711,0.591871,8.294230,9.086553,6.475515
1,LUAD,0.0,0.592732,1.588421,7.586157,9.623011,0.0,6.816049,0.000000,0.0,...,0.000000,10.406705,10.795447,8.007330,11.330637,5.041931,1.327170,6.344054,9.532814,4.216191
2,PRAD,0.0,3.511759,4.327199,6.881787,9.870730,0.0,6.972130,0.452595,0.0,...,0.000000,11.033850,10.292609,6.580491,11.152196,0.000000,0.000000,4.575712,10.485900,4.499029
3,PRAD,0.0,3.663618,4.507649,6.659068,10.196184,0.0,7.843375,0.434882,0.0,...,0.000000,11.137958,10.401871,6.259530,11.640385,3.068258,0.000000,5.421661,9.660732,6.612882
4,BRCA,0.0,2.655741,2.821547,6.539454,9.738265,0.0,6.566967,0.360982,0.0,...,0.360982,10.757932,9.944203,8.311430,11.275572,1.580097,0.000000,6.935271,10.425698,5.104899
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
796,BRCA,0.0,1.865642,2.718197,7.350099,10.006003,0.0,6.764792,0.496922,0.0,...,0.000000,10.321872,10.039084,8.837543,10.633331,3.205502,1.793854,7.713613,10.583938,5.444624
797,LUAD,0.0,3.942955,4.453807,6.346597,10.056868,0.0,7.320331,0.000000,0.0,...,0.000000,11.981107,11.459416,7.884439,11.056224,0.000000,0.000000,5.990202,9.996982,6.371409
798,COAD,0.0,3.249582,3.707492,8.185901,9.504082,0.0,7.536589,1.811101,0.0,...,5.574986,10.418865,11.101621,8.423078,10.898594,8.869415,6.318056,7.973887,9.976779,5.552949
799,PRAD,0.0,2.590339,2.787976,7.318624,9.987136,0.0,9.213464,0.000000,0.0,...,0.000000,11.102120,11.776721,7.219536,11.921294,4.545641,0.995304,6.921472,9.231459,5.223179


In [ ]:
X = genetics.drop(columns="cancer", axis = 1)
y = genetics["cancer"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.20,
    random_state = 42,
)

numeric_transformer = make_pipeline(SimpleImputer(strategy = 'mean'), StandardScaler())
numeric_features = X_train.select_dtypes(include = ['int64', 'float64']).columns
preprocessor = make_column_transformer((numeric_transformer, numeric_features))

In [9]:
model = make_pipeline(preprocessor, HistGradientBoostingClassifier())
model.fit(X_train, y_train)

,steps,"[('columntransformer', ...), ('histgradientboostingclassifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('pipeline', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [10]:
accuracy_score(y_test, model.predict(X_test))

1.0

In [11]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test, model.predict(X_test))


array([[61,  0,  0,  0,  0],
       [ 0, 17,  0,  0,  0],
       [ 0,  0, 25,  0,  0],
       [ 0,  0,  0, 29,  0],
       [ 0,  0,  0,  0, 29]])

In [12]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    model,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

cv_scores, cv_scores.mean()


(array([0.99378882, 1.        , 1.        , 0.99375   , 1.        ]),
 np.float64(0.9975077639751554))

### Data

The underlying source of this data is the The Cancer Genome Atlas Pan-Cancer Analysis Project. The data was accessed via synapse.org. The specific data for this model was collected and modified based on a submission to the UCI Irvine Machine Learning Repository.

- https://archive.ics.uci.edu/dataset/401/gene+expression+cancer+rna+seq

<u>**Data Dictionary for reference**</u>

##### Response

**`cancer`**

- `[object]` the clinically determined cancer type, one of:
    - `BRCA`: Breast Invasive Carcinoma
    - `PRAD`: Prostate Adenocarcinoma
    - `KIRC`: Kidney Renal Clear Cell Carcinoma
    - `LUAD`: Lung Adenocarcinoma
    - `COAD`: Colon Adenocarcinoma

##### Features

**`gene_####`**

- `[float64]` gene expression (for gene number `####` in the dataset) quantification as measured by an [Illumina HiSeq](https://www.illumina.com/content/dam/illumina-support/documents//products/datasheets/datasheet_hiseq_systems.pdf) platform
